# Bluechip Hackaton

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/bluechip-summit-credit-worthiness-prediction/Variable_def.xlsx
/kaggle/input/bluechip-summit-credit-worthiness-prediction/Sample Submission.csv
/kaggle/input/bluechip-summit-credit-worthiness-prediction/Train.csv
/kaggle/input/bluechip-summit-credit-worthiness-prediction/Test.csv


In [2]:
sample_data = pd.read_csv('/kaggle/input/bluechip-summit-credit-worthiness-prediction/Sample Submission.csv')
sample_data

,ID,Loan_Status
0,70607,NaN
1,58412,NaN
2,88755,NaN
3,97271,NaN
4,70478,NaN
...,...,...
2523,15578,NaN
2524,87689,NaN
2525,42584,NaN
2526,44709,NaN


In [3]:
train_data = pd.read_csv('/kaggle/input/bluechip-summit-credit-worthiness-prediction/Train.csv')
test_data = pd.read_csv('/kaggle/input/bluechip-summit-credit-worthiness-prediction/Test.csv')

# Display basic information and a preview of the data
train_data_info = train_data.info(), train_data.head()
test_data_info = test_data.info(), test_data.head()

train_data_info, test_data_info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5898 entries, 0 to 5897
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID                 5898 non-null   int64  
 1   Loan_ID            5898 non-null   object 
 2   Gender             5898 non-null   int64  
 3   Married            5898 non-null   int64  
 4   Dependents         5898 non-null   object 
 5   Education          5898 non-null   int64  
 6   Self_Employed      5898 non-null   int64  
 7   ApplicantIncome    5898 non-null   int64  
 8   CoapplicantIncome  5898 non-null   float64
 9   LoanAmount         5898 non-null   int64  
 10  Loan_Amount_Term   5898 non-null   int64  
 11  Credit_History     5898 non-null   int64  
 12  Property_Area      5898 non-null   int64  
 13  Loan_Status        5898 non-null   int64  
 14  Total_Income       5898 non-null   int64  
dtypes: float64(1), int64(12), object(2)
memory usage: 691.3+ KB
<class 'pand

((None,
        ID   Loan_ID  Gender  Married Dependents  Education  Self_Employed  \
  0  74768  LP002231       1        1          0          1              0   
  1  79428  LP001448       1        1          0          0              0   
  2  70497  LP002231       0        0          0          0              0   
  3  87480  LP001385       1        1          0          0              0   
  4  33964  LP002231       1        1          1          0              0   
  
     ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
  0             8328           0.000000          17               363   
  1              150        3857.458782         188               370   
  2             4989         314.472511          17               348   
  3              150           0.000000         232               359   
  4             8059           0.000000          17               372   
  
     Credit_History  Property_Area  Loan_Status  Total_Income  
  0             

In [4]:
# Display missing values
print(train_data.isnull().sum())
print(test_data.isnull().sum())

ID                   0
Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
Total_Income         0
dtype: int64
ID                   0
Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Total_Income         0
dtype: int64


In [5]:
# Convert the 'Dependents' column to a numeric type, handling possible object entries
train_data['Dependents'] = pd.to_numeric(train_data['Dependents'], errors='coerce')
test_data['Dependents'] = pd.to_numeric(test_data['Dependents'], errors='coerce')

In [6]:
binary_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Credit_History']
label_enc = LabelEncoder()

for col in binary_cols:
    train_data[col] = label_enc.fit_transform(train_data[col])
    test_data[col] = label_enc.transform(test_data[col])

print("Processed train and test data columns.")

Processed train and test data columns.


In [7]:
# Separating features and target variable
X = train_data.drop(['Loan_ID', 'ID', 'Loan_Status'], axis = 1)
y = train_data['Loan_Status']

# For test data, remove Loan_ID and ID
X_test = test_data.drop(['Loan_ID', 'ID'], axis = 1)

In [8]:
# Splitting data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 42)


In [9]:
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

# Impute missing values in X_train
imputer = SimpleImputer(strategy="mean")  # You can also use "median" or "most_frequent" based on the data
X_train_imputed = imputer.fit_transform(X_train)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_imputed, y_train)

print("SMOTE applied successfully.")

SMOTE applied successfully.


In [10]:
# Convert X_train_smote to DataFrame with original feature names (assuming X_train has these feature names)
X_train_smote_df = pd.DataFrame(X_train_smote, columns=X_train.columns)
X_val_df = pd.DataFrame(X_val, columns=X_train.columns)
X_test_df = pd.DataFrame(X_test, columns=X_train.columns)

# Scale the features
scaler = StandardScaler()

# Fit the scaler on the SMOTE-processed training set and transform all sets
X_train_smote_scaled = scaler.fit_transform(X_train_smote_df)
X_val_scaled = scaler.transform(X_val_df)
X_test_scaled = scaler.transform(X_test_df)

print("Features scaled successfully.")


Features scaled successfully.


In [11]:
# Initialize and train the Logistic Regression model
log_reg = LogisticRegression(max_iter = 1000, random_state = 42)
log_reg.fit(X_train_smote, y_train_smote)


LogisticRegression(max_iter=1000, random_state=42)

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Define the parameter grid for GridSearchCV
param_grid = {'log_reg__C': [0.1, 1, 10, 100], 'log_reg__solver': ['liblinear', 'lbfgs']}

# Create a pipeline to handle imputation and scaling, followed by logistic regression
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  # Impute missing values
    ('scaler', StandardScaler()),  # Scale the data
    ('log_reg', LogisticRegression(max_iter=1000, random_state=42))  # Logistic Regression model
])

# Initialize GridSearchCV with the pipeline
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_smote, y_train_smote)

# Get the best estimator and parameters
best_log_reg = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

# Re-evaluate with best parameters
y_val_pred = best_log_reg.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print("Classification Report:\n", classification_report(y_val, y_val_pred))


Best parameters: {'log_reg__C': 0.1, 'log_reg__solver': 'liblinear'}
Validation Accuracy: 0.523728813559322
Confusion Matrix:
 [[ 93  94]
 [468 525]]
Classification Report:
               precision    recall  f1-score   support

           0       0.17      0.50      0.25       187
           1       0.85      0.53      0.65       993

    accuracy                           0.52      1180
   macro avg       0.51      0.51      0.45      1180
weighted avg       0.74      0.52      0.59      1180



/opt/conda/lib/python3.10/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(


In [13]:
from sklearn.impute import SimpleImputer
import pandas as pd

# Ensure X_train_smote and X_test are DataFrames before imputing
X_train_smote = pd.DataFrame(X_train_smote, columns=X_train.columns)
X_test = pd.DataFrame(X_test, columns=X_train.columns)

# Impute missing values in X_train_smote and X_test
imputer = SimpleImputer(strategy="mean")
X_train_smote = pd.DataFrame(imputer.fit_transform(X_train_smote), columns=X_train.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_train.columns)

# Now make predictions with the adjusted test set
y_test_pred = best_log_reg.predict_proba(X_test)[:, 1]  # Probability for class 1 (approved)

# Create a DataFrame for submission
submission = pd.DataFrame({'ID': test_data['ID'], 'TARGET': y_test_pred})

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("Submission file created.")


Submission file created.


/opt/conda/lib/python3.10/site-packages/sklearn/base.py:432: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(
